<a href="https://colab.research.google.com/github/bhagyoday-j/ML-Assignments/blob/main/TY-open-elective/Assingments/5_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Assignment : 5***
Web Page Phishig Detection

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("peyamowar/phishing-and-benign-websites")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'phishing-and-benign-websites' dataset.
Path to dataset files: /kaggle/input/phishing-and-benign-websites


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import os

# The path to the dataset was printed in the previous cell
dataset_path = '/root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1'

print(f"Attempting to load data from: {dataset_path}")

try:
    file_name = 'phishing_and_benign_websites.csv'
    full_file_path = os.path.join(dataset_path, file_name)

    print(f"\nAttempting to load data from: {full_file_path}")
    df = pd.read_csv(full_file_path)
    print("Dataset loaded successfully!")
    print("Shape of the dataset:", df.shape)
    print("First 5 rows:\n", df.head())
    print("Column information:\n", df.info())

    # Correctly identify target variable and features
    # Based on df.head() and df.info(), 'Label' is the target. 'URLs' is the feature.
    y = df['Label']
    print("Target variable 'Label' identified.")

    # Encode target variable
    global label_encoder # Declare as global for later use in deployment
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    print(f"Target variable encoded. Original classes: {label_encoder.classes_}")

    # Feature Engineering: Extracting simple numerical features from URLs
    # For a basic Logistic Regression model, let's start with URL length as a feature.
    # More sophisticated features would involve parsing URLs (e.g., domain length, path length, number of dots, etc.)
    df['url_length'] = df['URLs'].apply(len)
    X = df[['url_length']] # X should be a DataFrame or 2D array for sklearn models
    print(f"Features created: {list(X.columns)}")

    global feature_names # Declare as global for later use in deployment
    feature_names = X.columns.tolist() # Store feature names for consistency during inference

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

    print(f"Data split: X_train shape {X_train.shape}, X_test shape {X_test.shape}")

    # Initialize and train Logistic Regression model
    global trained_model # Declare as global for later use in deployment
    trained_model = LogisticRegression(max_iter=1000, solver='liblinear')
    trained_model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = trained_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

    print(f"\nLogistic Regression Model Accuracy: {accuracy:.4f}")
    print("\nClassification Report:\n", report)

    print("\nModel training complete. The 'trained_model' object is now available for deployment.")

except FileNotFoundError:
    print(f"Error: The file {full_file_path} was not found. Please check the dataset path and file name.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempting to load data from: /root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1

Attempting to load data from: /root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1/phishing_and_benign_websites.csv
Dataset loaded successfully!
Shape of the dataset: (38800, 2)
First 5 rows:
                                           URLs   Label
0                     http://www.wmmayhem.com/  Benign
1  http://www.ballymenaunitedyouthacademy.com/  Benign
2              http://www.brusselsgaybars.com/  Benign
3          http://www.sportsbettingtennis.net/  Benign
4                         http://www.i29.mobi/  Benign
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38800 entries, 0 to 38799
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URLs    38800 non-null  object
 1   Label   38800 non-null  object
dtypes: object(2)
memory usage: 606.4+ KB
Column information:
 None
Target variable

### Setting up FastAPI and Ngrok

Since you requested FastAPI, I will set up a new application using FastAPI and expose it with `pyngrok`. This replaces the previous Flask attempt.

First, let's install the required libraries: `fastapi`, `uvicorn` (an ASGI server for FastAPI), and `pyngrok` (to create a public URL).

In [3]:
# Install FastAPI, Uvicorn, and Pyngrok
!pip install fastapi uvicorn pyngrok

### FastAPI Application Definition

Now, let's define the FastAPI application. It will have a `/predict` endpoint that takes a URL, extracts its length, and uses the pre-trained Logistic Regression model to classify it as 'Benign' or 'Phishing'.

Make sure to provide your `NGROK_AUTH_TOKEN` in Colab secrets. You can get one from [ngrok.com](https://ngrok.com/).



In [4]:
from fastapi import FastAPI, Request
from pydantic import BaseModel
from typing import Dict
import time
import sys
import os
import pandas as pd
from pyngrok import ngrok, conf
from google.colab import userdata
from fastapi.responses import HTMLResponse # Import HTMLResponse

# Initialize FastAPI app
app = FastAPI()

# Helper function to get current memory usage (in MB) for a given object
def get_memory_usage(obj):
    return sys.getsizeof(obj) / (1024 * 1024)

# Pydantic model for request body validation
class URLItem(BaseModel):
    url: str

@app.get('/')
async def home():
    return {"message": "Model API is running! Send POST requests to /predict."}

@app.post('/predict')
async def predict(item: URLItem):
    start_time = time.time()
    url = item.url

    try:
        # Feature extraction - must be consistent with training
        url_length = len(url)
        # Ensure features is a DataFrame with the correct column name
        features = pd.DataFrame([[url_length]], columns=feature_names)

        # Make prediction
        prediction_encoded = trained_model.predict(features)[0]
        prediction_proba = trained_model.predict_proba(features)[0]

        # Decode prediction
        predicted_label = label_encoder.inverse_transform([prediction_encoded])[0]

        # Prepare probabilities for output
        probabilities = {label: prob for label, prob in zip(label_encoder.classes_, prediction_proba)}

        end_time = time.time()
        inference_time = (end_time - start_time) * 1000 # in milliseconds

        # Log metrics
        model_memory_usage = get_memory_usage(trained_model)

        return {
            'prediction': predicted_label,
            'probabilities': probabilities,
            'inference_time_ms': inference_time,
            'model_memory_usage_mb': model_memory_usage
        }
    except Exception as e:
        return {'error': str(e), 'message': 'An error occurred during prediction.'}, 500

# New endpoint to serve the UI
@app.get('/ui', response_class=HTMLResponse)
async def serve_ui():
    # HTML and JavaScript for the UI - this content is taken from cell 0589b940
    ui_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Phishing Detection UI</title>
    <style>
        body {{ font-family: sans-serif; margin: 20px; background-color: #f0f2f5; color: #333; }}
        .container {{ max-width: 600px; margin: 0 auto; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); background-color: #ffffff; }}
        h2 {{ color: #0056b3; text-align: center; margin-bottom: 25px; }}
        label {{ display: block; margin-bottom: 8px; font-weight: bold; color: #555; }}
        input[type=\"text\"] {{ width: calc(100% - 22px); padding: 12px; margin-bottom: 20px; border: 1px solid #ccc; border-radius: 8px; font-size: 16px; box-sizing: border-box; }}
        button {{ background-color: #007bff; color: white; padding: 12px 25px; border: none; border-radius: 8px; cursor: pointer; font-size: 17px; transition: background-color 0.3s ease; display: block; width: 100%; }}
        button:hover {{ background-color: #0056b3; }}
        #result {{ margin-top: 25px; padding: 18px; background-color: #e9ecef; border-radius: 8px; border: 1px solid #dee2e6; font-size: 16px; line-height: 1.6; }}
        #result p {{ margin: 0 0 8px 0; }}
        #result strong {{ color: #0056b3; }}
        .spinner {{ display: none; margin: 15px auto; border: 4px solid rgba(0, 0, 0, 0.1); border-left-color: #007bff; border-radius: 50%; width: 30px; height: 30px; animation: spin 1s linear infinite; }}
        @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}
        .error {{ color: #dc3545; font-weight: bold; }}
    </style>
</head>
<body>

<div class=\"container\">
    <h2>Phishing URL Detector</h2>
    <label for=\"urlInput\">Enter URL:</label>
    <input type=\"text\" id=\"urlInput\" placeholder=\"e.g., http://example.com/malicious-link\">
    <button onclick=\"predictUrl()\">Predict</button>
    <div class=\"spinner\" id=\"loadingSpinner\"></div>
    <div id=\"result\"></div>
</div>

<script>
    // The apiUrl needs to be relative or derived from the current host
    // When served via ngrok, the client-side JavaScript should use a relative path
    const apiUrl = window.location.origin + '/predict';
    const urlInput = document.getElementById('urlInput');
    const resultDiv = document.getElementById('result');
    const loadingSpinner = document.getElementById('loadingSpinner');

    async function predictUrl() {{
        const url = urlInput.value;
        if (!url) {{
            resultDiv.innerHTML = '<p class=\"error\">Please enter a URL.</p>';
            return;
        }}

        resultDiv.innerHTML = '';
        loadingSpinner.style.display = 'block';

        try {{
            const response = await fetch(apiUrl, {{
                method: 'POST',
                headers: {{
                    'Content-Type': 'application/json'
                }},
                body: JSON.stringify({{ url: url }})
            }});

            const data = await response.json();

            if (response.ok) {{
                let output = `<p><strong>Prediction:</strong> ${{data.prediction}}</p>`;
                output += `<p><strong>Probabilities:</strong></p><ul>`;
                for (const label in data.probabilities) {{
                    output += `<li>${{label}}: ${{ (data.probabilities[label] * 100).toFixed(2) }}%</li>`;
                }}
                output += `</ul>`;
                output += `<p><strong>Inference Time:</strong> ${{ data.inference_time_ms.toFixed(2) }} ms</p>`;
                resultDiv.innerHTML = output;
            }} else {{
                resultDiv.innerHTML = `<p class=\"error\">Error: ${{ data.message || data.error || 'Unknown error' }}</p>`;
            }}
        }} catch (error) {{
            console.error('Fetch error:', error);
            resultDiv.innerHTML = `<p class=\"error\">Failed to connect to the API. Make sure the FastAPI server is running.</p>`;
        }} finally {{
            loadingSpinner.style.display = 'none';
        }}
    }}
</script>

</body>
</html>
"""
    return HTMLResponse(content=ui_html, status_code=200)

print("FastAPI app with prediction endpoint '/predict' and UI endpoint '/ui' defined.")

FastAPI app with prediction endpoint '/predict' and UI endpoint '/ui' defined.


### Running FastAPI with Uvicorn and Ngrok

Now, I will run the FastAPI application using `uvicorn` and expose it to the internet using `pyngrok`. This will provide a public URL you can use to send requests.

**Note**: The "good UI" part typically involves building a separate frontend application (e.g., using HTML, CSS, JavaScript frameworks like React/Vue/Angular) that interacts with this API. This setup provides the API backend.

In [5]:
# Get Ngrok Auth Token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Start ngrok tunnel
# Ensure uvicorn is running on port 8000
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# Run the FastAPI application using uvicorn in a background thread
import uvicorn
import threading

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.daemon = True # Allow the main program to exit even if the thread is still running
uvicorn_thread.start()

print("FastAPI application is running. Use the public URL above to send requests.")
print("Example request (using `curl` in your terminal):")
print(f"curl -X POST -H \"Content-Type: application/json\" -d '{{\"url\": \"http://www.google.com\"}}' {public_url}/predict")

Public URL: NgrokTunnel: "https://gutter-scarf-synthesis.ngrok-free.dev" -> "http://localhost:8000"
FastAPI application is running. Use the public URL above to send requests.
Example request (using `curl` in your terminal):
curl -X POST -H "Content-Type: application/json" -d '{"url": "http://www.google.com"}' NgrokTunnel: "https://gutter-scarf-synthesis.ngrok-free.dev" -> "http://localhost:8000"/predict


### Simple Web UI for Prediction

The UI is now served directly by the FastAPI application. You can access it through your public ngrok URL. No need to run the UI cell separately in Colab anymore.

After running the FastAPI server cell, open your ngrok public URL (e.g., `https://xxxxxx.ngrok-free.dev`) in your browser and append `/ui` to it (e.g., `https://xxxxxx.ngrok-free.dev/ui`).

In [6]:
# This cell is no longer needed to display the UI within Colab,
# as the UI is now served by the FastAPI application itself.
# You can safely remove or ignore this cell after the FastAPI app is updated and running.
# from IPython.display import HTML

# # Ensure public_url is available from the previous ngrok cell
# # If you rerun this cell, make sure public_url is still valid

# # Extract the string representation of the public_url, removing NgrokTunnel: and arrow if present
# if isinstance(public_url, str):
#     api_url = public_url.split(' ')[-1].replace('"', '') # Assumes format 'NgrokTunnel: "url" -> "localhost"'
# else:
#     # If public_url is an object, try to get its URL attribute
#     api_url = public_url.public_url

# # HTML and JavaScript for the UI
# ui_html = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Phishing Detection UI</title>
#     <style>
#         body {{ font-family: sans-serif; margin: 20px; background-color: #f0f2f5; color: #333; }}
#         .container {{ max-width: 600px; margin: 0 auto; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); background-color: #ffffff; }}
#         h2 {{ color: #0056b3; text-align: center; margin-bottom: 25px; }}
#         label {{ display: block; margin-bottom: 8px; font-weight: bold; color: #555; }}
#         input[type=\"text\"] {{ width: calc(100% - 22px); padding: 12px; margin-bottom: 20px; border: 1px solid #ccc; border-radius: 8px; font-size: 16px; box-sizing: border-box; }}
#         button {{ background-color: #007bff; color: white; padding: 12px 25px; border: none; border-radius: 8px; cursor: pointer; font-size: 17px; transition: background-color 0.3s ease; display: block; width: 100%; }}
#         button:hover {{ background-color: #0056b3; }}
#         #result {{ margin-top: 25px; padding: 18px; background-color: #e9ecef; border-radius: 8px; border: 1px solid #dee2e6; font-size: 16px; line-height: 1.6; }}
#         #result p {{ margin: 0 0 8px 0; }}
#         #result strong {{ color: #0056b3; }}
#         .spinner {{ display: none; margin: 15px auto; border: 4px solid rgba(0, 0, 0, 0.1); border-left-color: #007bff; border-radius: 50%; width: 30px; height: 30px; animation: spin 1s linear infinite; }}
#         @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}
#         .error {{ color: #dc3545; font-weight: bold; }}
#     </style>
# </head>
# <body>

# <div class=\"container\">
#     <h2>Phishing URL Detector</h2>
#     <label for=\"urlInput\">Enter URL:</label>
#     <input type=\"text\" id=\"urlInput\" placeholder=\"e.g., http://example.com/malicious-link\">
#     <button onclick=\"predictUrl()\">Predict</button>
#     <div class=\"spinner\" id=\"loadingSpinner\"></div>
#     <div id=\"result\"></div>
# </div>

# <script>
#     const apiUrl = '"""" + api_url + """" + '/predict';
#     const urlInput = document.getElementById('urlInput');
#     const resultDiv = document.getElementById('result');
#     const loadingSpinner = document.getElementById('loadingSpinner');

#     async function predictUrl() {
#         const url = urlInput.value;
#         if (!url) {
#             resultDiv.innerHTML = '<p class=\"error\">Please enter a URL.</p>';
#             return;
#         }

#         resultDiv.innerHTML = '';
#         loadingSpinner.style.display = 'block';

#         try {
#             const response = await fetch(apiUrl, {
#                 method: 'POST',
#                 headers: {
#                     'Content-Type': 'application/json'
#                 },
#                 body: JSON.stringify({ url: url })
#             });

#             const data = await response.json();

#             if (response.ok) {
#                 let output = `<p><strong>Prediction:</strong> ${data.prediction}</p>`;
#                 output += `<p><strong>Probabilities:</strong></p><ul>`;
#                 for (const label in data.probabilities) {
#                     output += `<li>${label}: ${(data.probabilities[label] * 100).toFixed(2)}%</li>`;
#                 }
#                 output += `</ul>`;
#                 output += `<p><strong>Inference Time:</strong> ${data.inference_time_ms.toFixed(2)} ms</p>`;
#                 resultDiv.innerHTML = output;
#             } else {
#                 resultDiv.innerHTML = `<p class=\"error\">Error: ${data.message || data.error || 'Unknown error'}</p>`;
#             }
#         } catch (error) {
#             console.error('Fetch error:', error);
#             resultDiv.innerHTML = `<p class=\"error\">Failed to connect to the API. Make sure the FastAPI server is running.</p>`;
#         } finally {
#             loadingSpinner.style.display = 'none';
#         }
#     }
# </script>

# </body>
# </html>
# """

# display(HTML(ui_html))

In [7]:
# from flask import Flask, request, jsonify
# from flask_ngrok import run_with_ngrok
# import time
# import sys
# import os

# # Initialize Flask app
# app = Flask(__name__)
# run_with_ngrok(app) # Start ngrok when app is run

# # Store memory usage and inference times
# # This can be made more robust for concurrent requests if needed, but for simple evaluation it's fine.
# memory_usage_log = []
# inference_time_log = []

# # Helper function to get current memory usage (in MB)
# def get_memory_usage():
#     return sys.getsizeof(trained_model) / (1024 * 1024) # Placeholder, ideally measure process memory

# @app.route('/')
# def home():
#     return "Model API is running! Send POST requests to /predict."

# @app.route('/predict', methods=['POST'])
# def predict():
#     start_time = time.time()
#     data = request.get_json(force=True)

#     if 'url' not in data:
#         return jsonify({'error': 'Missing "url" field in request.'}), 400

#     url = data['url']

#     try:
#         # Feature extraction - must be consistent with training
#         url_length = len(url)
#         features = pd.DataFrame([[url_length]], columns=feature_names)

#         # Make prediction
#         prediction_encoded = trained_model.predict(features)[0]
#         prediction_proba = trained_model.predict_proba(features)[0]

#         # Decode prediction
#         predicted_label = label_encoder.inverse_transform([prediction_encoded])[0]

#         # Prepare probabilities for output
#         probabilities = {label: prob for label, prob in zip(label_encoder.classes_, prediction_proba)}

#         end_time = time.time()
#         inference_time = (end_time - start_time) * 1000 # in milliseconds

#         # Log metrics (simplistic for demonstration)
#         memory_usage = get_memory_usage()
#         inference_time_log.append(inference_time)
#         memory_usage_log.append(memory_usage)

#         return jsonify({
#             'prediction': predicted_label,
#             'probabilities': probabilities,
#             'inference_time_ms': inference_time,
#             'model_memory_usage_mb': memory_usage # This is just the model object size, not process memory
#         })
#     except Exception as e:
#         return jsonify({'error': str(e)}), 500


# # This part needs to be run in a separate cell or handled carefully in Colab if not using run_with_ngrok properly
# # For simplicity, run_with_ngrok already starts the app in a new thread.
# # app.run()

# print("Flask app with prediction endpoint '/predict' is ready. Ngrok tunnel will start upon execution of this cell.")
# print("You can send POST requests with JSON payload like: {'url': 'http://example.com'}")

In [8]:
!pip install flask_ngrok

INFO:     Started server process [13222]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
